In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import sklearn

In [ ]:
investor_data = pd.read_csv("C:\\Users\\roni2\\OneDrive\\Documents\\investor_data_2.csv")
investor_data.head(3)

,investor,commit,deal_size,invite,rating,int_rate,covenants,total_fees,fee_share,prior_tier,invite_tier,tier_change,fee_percent,invite_percent
0,Goldman Sachs,Commit,300,40,2,Market,2,30,0.0,Participant,Bookrunner,Promoted,0.000000,0.133333
1,Deutsche Bank,Decline,1200,140,2,Market,2,115,20.1,Bookrunner,Participant,Demoted,0.174783,0.116667
2,Bank of America,Commit,900,130,3,Market,2,98,24.4,Bookrunner,Bookrunner,NaN,0.248980,0.144444


In [ ]:
# Drop columns not needed for classification
investor_data = investor_data.drop(['invite_tier', 'invite', 'fee_share'], axis = 1)
investor_data.shape

(7233, 11)

In [ ]:
# Change categorical variables into numerical variables 
investor_data = pd.get_dummies(investor_data, dtype=int)
investor_data.shape

(7233, 20)

In [9]:
investor_data.head(1)

,deal_size,rating,covenants,total_fees,fee_percent,invite_percent,investor_Bank of America,investor_Deutsche Bank,investor_Goldman Sachs,investor_MUFG Union,investor_Wells Fargo,commit_Commit,commit_Decline,int_rate_Above,int_rate_Below,int_rate_Market,prior_tier_Bookrunner,prior_tier_Participant,tier_change_Demoted,tier_change_Promoted
0,300,2,2,30,0.0,0.133333,0,0,1,0,0,1,0,0,0,1,0,1,0,1


In [ ]:
# Drop one of the commit columns for single dataset to explain commit status
investor_data = investor_data.drop('commit_Commit', axis=1)
investor_data.shape

(7233, 19)

In [ ]:
# Define target and inputs
target = investor_data.commit_Decline
inputs = investor_data.drop('commit_Decline', axis=1)

In [ ]:
from sklearn.model_selection import train_test_split

# Stratified split to maintain class proportions in train and test sets
split_list = train_test_split (inputs, target, test_size=0.2, random_state=1, stratify=investor_data.commit_Decline)

In [17]:
for item in split_list:
    print(item.shape)

(5786, 18)
(1447, 18)
(5786,)
(1447,)


In [ ]:
# Unpack split list
input_train, input_test, target_train, target_test = split_list

print(input_train.shape)
print(input_test.shape)
print(target_train.shape)
print(target_test.shape)

(5786, 18)
(1447, 18)
(5786,)
(1447,)


In [21]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
# Create pipelines for each model
pipelines = {
    'l1' : make_pipeline(StandardScaler(), LogisticRegression(penalty='l1', random_state=1, solver='liblinear')),
    'l2' : make_pipeline(StandardScaler(), LogisticRegression(penalty='l2', random_state=1, solver='liblinear')),
    'rf' : make_pipeline(StandardScaler(), RandomForestClassifier(random_state=1)),
    'gb' : make_pipeline(StandardScaler(), GradientBoostingClassifier(random_state=1))
}
                        

In [25]:
for key, value in pipelines.items():
    print(key, type(value))

l1 <class 'sklearn.pipeline.Pipeline'>
l2 <class 'sklearn.pipeline.Pipeline'>
rf <class 'sklearn.pipeline.Pipeline'>
gb <class 'sklearn.pipeline.Pipeline'>


In [ ]:
# Define hyperparameters for each regression model
l1_hyperparameters = {
    'logisticregression__C' : [0.1, 1, 10]  
}

l2_hyperparameters = {
    'logisticregression__C' : [0.1, 1, 10]
}

rf_hyperparameters = {
    'randomforestclassifier__n_estimators' : [100, 200],
    'randomforestclassifier__max_features' : [None, 0.3, 0.6]
}

gb_hyperparameters = {
    'gradientboostingclassifier__n_estimators' : [100, 200],
    'gradientboostingclassifier__learning_rate' : [0.05, 0.1, 0.2],
    'gradientboostingclassifier__max_depth' : [1, 3, 5]
}



In [ ]:
# Combine hyperparameters into single dictionary
hyperparameters = {
    'l1' : l1_hyperparameters,
    'l2' : l2_hyperparameters,
    'rf' : rf_hyperparameters,
    'gb' : gb_hyperparameters
}

In [ ]:
# Check that all models have hyperparameter grids defined

for key in ['l1', 'l2', 'rf', 'gb']:
    if key in hyperparameters:
        if type(hyperparameters[key]) is dict:
            print (key, 'was found, and it is a grid.')
        else:
            print (key, 'was found, but it is not a grid.')
    else:
        print(key, 'was not found')

l1 was found, and it is a grid.
l2 was found, and it is a grid.
rf was found, and it is a grid.
gb was found, and it is a grid.


In [ ]:
from sklearn.model_selection import GridSearchCV

# Create a dictionary to hold the model objects after GridSearchCV
models = {}

for key in pipelines.keys():
    models[key] = GridSearchCV(pipelines[key], hyperparameters[key], cv=5)

models.keys()

dict_keys(['l1', 'l2', 'rf', 'gb'])

In [ ]:
# Train and tune each model in the models dictionary
for key in models:
    models[key].fit(input_train,target_train)
    print(key, 'is trained and tuned.')

l1 is trained and tuned.
l2 is trained and tuned.
rf is trained and tuned.
gb is trained and tuned.


In [ ]:
from sklearn.metrics import roc_curve, auc

# Evaluate each model using AUROC

for trained_model in models.keys():
    preds = models[trained_model].predict(input_test)
    fpr, tpr, thresholds = roc_curve(target_test, preds)
    print(trained_model)
    print('AUROC =', round(auc(fpr, tpr), 4))
    print('---')

l1
AUROC = 0.9522
---
l2
AUROC = 0.9518
---
rf
AUROC = 0.9595
---
gb
AUROC = 0.9691
---
